# 04 - Dataset ampliado y modelo final
**Maestría en IA - UDIT** · Lenguajes de Programación para IA · Prof. Carlos Cob
Alumno: Santiago Toledo Berral (STB)

Partiendo del Ridge + TF-IDF (SMAPE ~51%), intento mejorar con:
1. Separación train / validation / test (70/15/15) agrupada por imagen. El test se usa una sola vez al final.
2. Variables nuevas: un clustering del tipo de producto y precios típicos por grupo y por marca.
3. Selección de variables.
4. LightGBM que recibe también la predicción del Ridge (stacking).

El código reutilizable está en `src/artifacts/limpieza.py` y `src/artifacts/features.py`.

In [1]:
import json
import time
from datetime import date
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

import sys
sys.path.insert(0, "../src")
from artifacts import features as sf, limpieza as stb

pio.renderers.default = "plotly_mimetype+notebook_connected"
pio.templates.default = "plotly_white"
AZUL, ROJO, GRIS = "#2a78d6", "#c23a3a", "#898781"
DATA_DIR, MODELS_DIR = Path("../data"), Path("../models")
T0 = time.time()

## 1. Limpieza y separación

In [2]:
df, _ = stb.limpiar_dataset(pd.read_csv(DATA_DIR / "train_full.csv"))
df["split"] = sf.crear_split(df["image_group"], frac_val=0.15, frac_test=0.15)
display(df.groupby("split")["price"].agg(["size", "median", "mean"]).round(2))

# test.csv de la competencia: no trae precio, solo lo uso para entrenar el clustering
RUTA_TEST_EXT = Path.home() / ".cache/kagglehub/datasets/satwiksps/fully-extracted-dataset/versions/1/test.csv"
if not RUTA_TEST_EXT.exists():
    import kagglehub
    RUTA_TEST_EXT = Path(kagglehub.dataset_download("satwiksps/fully-extracted-dataset", path="test.csv"))
nombres_ext = sf.features_basicas(pd.read_csv(RUTA_TEST_EXT)["catalog_content"])["nombre_producto"]

,size,median,mean
split,,,
test,11262,14.23,23.84
train,52481,14.00,23.63
validation,11230,14.00,23.53


## 2. Dataset ampliado
- Variables directas del texto: cantidad, pack, categoría, palabras clave, etc.
- `cluster_producto`: KMeans de 60 grupos sobre el nombre (TF-IDF + SVD). No usa el precio.
- Precios típicos del cluster y de la marca: medianas del log-precio calculadas solo con train y fuera de fold para no filtrar el target.

In [3]:
es_tr, es_va, es_te = (df["split"] == s for s in ["train", "validation", "test"])
f_basicas = sf.features_basicas(df["catalog_content"]).set_axis(df.index)

codificador = sf.CodificadorAprendido(n_clusters=60)
enc_tr = codificador.fit_transform(f_basicas[es_tr], df.loc[es_tr, "log_price"], df.loc[es_tr, "image_group"],
                                   nombres_extra=nombres_ext).set_axis(df.index[es_tr])
enc_va = codificador.transform(f_basicas[es_va]).set_axis(df.index[es_va])
enc_te = codificador.transform(f_basicas[es_te]).set_axis(df.index[es_te])
f_aprendidas = pd.concat([enc_tr, enc_va, enc_te]).loc[df.index]

dataset = pd.concat([df[["sample_id", "split", "price", "log_price"]].rename(columns={"price": "precio", "log_price": "log_precio"}),
                     f_basicas, f_aprendidas], axis=1)
print(dataset.shape)

(74973, 39)


In [4]:
info = (pd.DataFrame({"cluster": f_aprendidas.loc[es_tr, "cluster_producto"], "precio": df.loc[es_tr, "price"]})
          .groupby("cluster")["precio"].median().sort_values().to_frame("precio_mediano"))
info["palabras"] = codificador.terminos_cluster(4)
ext = pd.concat([info.head(8), info.tail(8)])
fig = go.Figure(go.Bar(y=[f"#{c}: {p[:40]}" for c, p in zip(ext.index, ext["palabras"])], x=ext["precio_mediano"], orientation="h",
                       marker_color=[AZUL] * 8 + [ROJO] * 8))
fig.update_layout(title="Clusters más baratos y más caros", xaxis_title="Precio mediano ($)", height=550, margin=dict(l=280))
fig.show()

El clustering separa bien tipos de producto (sopas y conservas frente a café en grano o kits de regalo), que es justo lo que hace falta para interpretar la cantidad.

## 3. Selección de variables
1. Quito las casi constantes (>99% el mismo valor).
2. Quito una de cada par muy correlacionado (Spearman > 0,95).
3. Importancia por permutación en validation: desordeno cada variable y miro cuánto empeora el SMAPE.

In [5]:
CANDIDATAS = [c for c in dataset.columns if c not in {"sample_id", "split", "precio", "log_precio", "nombre_producto", "marca"}]
tr = dataset[es_tr]
quitar_var = [c for c in CANDIDATAS if tr[c].value_counts(normalize=True, dropna=False).iloc[0] > 0.99]

num = [c for c in CANDIDATAS if pd.api.types.is_numeric_dtype(tr[c]) and c not in quitar_var + ["cluster_producto"]]
corr = tr[num].corr(method="spearman")
quitar_corr = sorted({b for i, a in enumerate(num) for b in num[i + 1:] if abs(corr.loc[a, b]) > 0.95})

PRESELECCION = [c for c in CANDIDATAS if c not in quitar_var + quitar_corr]
print("Casi constantes:", quitar_var)
print("Redundantes:", quitar_corr)
print(f"{len(CANDIDATAS)} -> {len(PRESELECCION)} variables")

Casi constantes: []
Redundantes: ['es_pack', 'log_cantidad', 'log_cantidad_por_unidad', 'log_palabras', 'log_unidades_pack']
33 -> 28 variables


In [6]:
X_tr, y_tr, g_tr = df.loc[es_tr, "catalog_content"], df.loc[es_tr, "log_price"].to_numpy(), df.loc[es_tr, "image_group"]
X_va, y_va, precio_va = df.loc[es_va, "catalog_content"], df.loc[es_va, "log_price"].to_numpy(), df.loc[es_va, "price"].to_numpy()

modelo_pre = sf.ModeloPreciosV2(PRESELECCION)
modelo_pre.fit(X_tr, y_tr, g_tr, nombres_extra=nombres_ext, eval_set=(X_va, y_va))

# Permutación sobre la parte LightGBM
f_va = modelo_pre.preparar(X_va)
Xv = modelo_pre._tabla_lgbm(f_va, modelo_pre.ridge_.predict(modelo_pre._matriz_ridge(f_va)))
base = stb.smape(precio_va, np.expm1(modelo_pre.lgbm_.predict(Xv)))
grupos = {c: [c] for c in Xv.columns if not c.startswith("texto_svd_")}
grupos["texto_svd"] = [c for c in Xv.columns if c.startswith("texto_svd_")]

rng = np.random.default_rng(42)
importancia = {}
for nombre, cols in grupos.items():
    Xp = Xv.copy()
    Xp[cols] = Xp[cols].iloc[rng.permutation(len(Xp))].set_axis(Xp.index)
    importancia[nombre] = stb.smape(precio_va, np.expm1(modelo_pre.lgbm_.predict(Xp))) - base
importancia = pd.Series(importancia).sort_values()
print(f"SMAPE validation con {len(PRESELECCION)} variables: {base:.2f}  ({time.time() - T0:.0f}s)")

SMAPE validation con 28 variables: 46.95  (304s)


In [7]:
UMBRAL = 0.02
fig = go.Figure(go.Bar(y=importancia.index, x=importancia.clip(lower=1e-3), orientation="h",
                       marker_color=np.where(importancia > UMBRAL, AZUL, GRIS)))
fig.add_vline(x=UMBRAL, line_dash="dot")
fig.update_layout(title="Cuánto empeora el SMAPE al desordenar cada variable (gris = no aporta)",
                  xaxis=dict(title="Aumento del SMAPE (puntos, log)", type="log"), height=650, margin=dict(l=220))
fig.show()

SELECCIONADAS = [c for c in PRESELECCION if importancia.get(c, 0) > UMBRAL]
print(f"Seleccionadas ({len(SELECCIONADAS)}):", SELECCIONADAS)

Seleccionadas (20): ['categoria', 'familia_unidad', 'cantidad_estandar', 'unidades_pack', 'cantidad_por_unidad', 'n_caracteres', 'n_bullets', 'tiene_descripcion', 'palabras_nombre', 'ratio_mayusculas', 'ratio_digitos', 'es_regalo', 'es_granel', 'es_variedad', 'cluster_producto', 'precio_tipico_cluster', 'precio_por_unidad_cluster', 'precio_tipico_marca', 'frecuencia_marca', 'log_precio_esperado']


## 4. Comparación de modelos en validation
- M0: la mediana, como referencia.
- M1: Ridge v1 (TF-IDF del nombre + numéricas).
- M2: Ridge v2 (+ cluster x cantidad).
- M3 y M4: LightGBM con la predicción del Ridge como una variable más. En train esa predicción se calcula fuera de fold.

In [8]:
def metricas(nombre, pred_log, y_log, precio):
    pred = np.expm1(pred_log)
    return {"modelo": nombre, "SMAPE (%)": stb.smape(precio, pred), "MAE ($)": mean_absolute_error(precio, pred),
            "R² (log)": r2_score(y_log, pred_log)}

ridge_v1 = Pipeline([("features", FunctionTransformer(stb.construir_features)),
                     ("prep", ColumnTransformer([("num", StandardScaler(), stb.COLUMNAS_NUMERICAS),
                                                 ("cat", OneHotEncoder(handle_unknown="ignore"), stb.COLUMNAS_CATEGORICAS),
                                                 ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=3, sublinear_tf=True), "nombre")])),
                     ("modelo", Ridge(alpha=1.0))]).fit(X_tr.to_frame(), y_tr)

modelo_sel = sf.ModeloPreciosV2(SELECCIONADAS)
modelo_sel.fit(X_tr, y_tr, g_tr, nombres_extra=nombres_ext, eval_set=(X_va, y_va))
partes_pre, partes_sel = modelo_pre.predecir_partes(X_va), modelo_sel.predecir_partes(X_va)

comparacion = pd.DataFrame([
    metricas("M0 Mediana", np.full(len(y_va), np.median(y_tr)), y_va, precio_va),
    metricas("M1 Ridge v1", ridge_v1.predict(X_va.to_frame()), y_va, precio_va),
    metricas("M2 Ridge v2", partes_pre["ridge"], y_va, precio_va),
    metricas(f"M3 LightGBM ({len(PRESELECCION)} variables)", partes_pre["lgbm"], y_va, precio_va),
    metricas(f"M4 LightGBM ({len(SELECCIONADAS)} variables)", partes_sel["lgbm"], y_va, precio_va),
]).set_index("modelo")
print(f"({time.time() - T0:.0f}s)")
comparacion.round(3)

(478s)


,SMAPE (%),MAE ($),R² (log)
modelo,,,
M0 Mediana,72.213,16.361,-0.001
M1 Ridge v1,51.791,11.608,0.485
M2 Ridge v2,50.947,11.367,0.493
M3 LightGBM (28 variables),46.945,10.367,0.534
M4 LightGBM (20 variables),46.856,10.307,0.534


M3 y M4 quedan prácticamente iguales, así que me quedo con M4, que usa menos variables. Probé también promediar Ridge y LightGBM y no mejoraba, porque LightGBM ya recibe la predicción del Ridge.

## 5. Evaluación final en test
Reentreno con train + validation y evalúo una única vez en test.

In [9]:
FEATURES = modelo_sel.features_lgbm
N_ARBOLES = modelo_sel.lgbm_.best_iteration_
es_trva = es_tr | es_va
X_te, y_te, precio_te = df.loc[es_te, "catalog_content"], df.loc[es_te, "log_price"].to_numpy(), df.loc[es_te, "price"].to_numpy()

modelo_test = sf.ModeloPreciosV2(FEATURES, params_lgbm={**sf.PARAMS_LGBM, "n_estimators": int(N_ARBOLES * 1.15)})
modelo_test.fit(df.loc[es_trva, "catalog_content"], df.loc[es_trva, "log_price"], df.loc[es_trva, "image_group"],
                nombres_extra=nombres_ext)
partes_te = modelo_test.predecir_partes(X_te)
ridge_v1.fit(df.loc[es_trva, ["catalog_content"]], df.loc[es_trva, "log_price"])
pred_v1 = ridge_v1.predict(X_te.to_frame())

res_test = pd.DataFrame([
    metricas("Mediana", np.full(len(y_te), df.loc[es_trva, "log_price"].median()), y_te, precio_te),
    metricas("Ridge v1", pred_v1, y_te, precio_te),
    metricas("Ridge v2", partes_te["ridge"], y_te, precio_te),
    metricas("Modelo v2 (LightGBM)", partes_te["lgbm"], y_te, precio_te),
]).set_index("modelo")
print(f"({time.time() - T0:.0f}s)")
res_test.round(3)

(697s)


,SMAPE (%),MAE ($),R² (log)
modelo,,,
Mediana,73.233,16.779,-0.001
Ridge v1,51.280,11.876,0.498
Ridge v2,50.411,11.530,0.508
Modelo v2 (LightGBM),46.353,10.514,0.545


In [10]:
tramo = pd.cut(precio_te, [0, 5, 10, 20, 50, 100, np.inf], labels=["< $5", "$5-10", "$10-20", "$20-50", "$50-100", "> $100"])
pt = pd.DataFrame({"tramo": tramo, "real": precio_te, "v1": np.expm1(pred_v1), "v2": np.expm1(partes_te["lgbm"].to_numpy())})
pt = pt.groupby("tramo", observed=True).apply(lambda g: pd.Series({"Ridge v1": stb.smape(g["real"], g["v1"]),
                                                                  "Modelo v2": stb.smape(g["real"], g["v2"])}),
                                              include_groups=False)
fig = go.Figure([go.Bar(x=pt.index.astype(str), y=pt["Ridge v1"], name="Ridge v1", marker_color=GRIS),
                 go.Bar(x=pt.index.astype(str), y=pt["Modelo v2"], name="Modelo v2", marker_color=AZUL)])
fig.update_layout(title="SMAPE en test por tramo de precio", xaxis_title="Precio real", yaxis_title="SMAPE (%)")
fig.show()

## 6. Modelo final
Reentreno con el 100% de los datos y lo guardo en `models/stb_modelo_v2.joblib`, que es el que usa la API. La matriz SVD del texto se guarda en float16 para que el archivo pese la mitad; las predicciones no cambian de forma apreciable.

In [11]:
dataset[["sample_id", "split", "precio", "log_precio"] + list(FEATURES)].to_parquet(DATA_DIR / "dataset_seleccionado.parquet", index=False)

modelo_final = sf.ModeloPreciosV2(FEATURES, params_lgbm={**sf.PARAMS_LGBM, "n_estimators": int(N_ARBOLES * 1.2)})
modelo_final.fit(df["catalog_content"], df["log_price"], df["image_group"], nombres_extra=nombres_ext)
svd = modelo_final.cod_.svd_
svd.components_ = svd.components_.astype(np.float16)
joblib.dump(modelo_final, MODELS_DIR / "stb_modelo_v2.joblib", compress=3)

meta = {"fecha": date.today().isoformat(), "filas": len(df), "features_lgbm": list(FEATURES),
        "smape_validation": round(float(comparacion.iloc[4]["SMAPE (%)"]), 2),
        "metricas_test": res_test.loc["Modelo v2 (LightGBM)"].round(4).to_dict()}
(MODELS_DIR / "stb_modelo_v2.json").write_text(json.dumps(meta, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"models/stb_modelo_v2.joblib: {(MODELS_DIR / 'stb_modelo_v2.joblib').stat().st_size / 1e6:.0f} MB "
      f"| tiempo total {(time.time() - T0) / 60:.1f} min")

models/stb_modelo_v2.joblib: 43 MB | tiempo total 14.4 min


In [12]:
# Prueba de carga (igual que en la API)
cargado = joblib.load(MODELS_DIR / "stb_modelo_v2.joblib")
cargado.cod_.svd_.components_ = cargado.cod_.svd_.components_.astype(np.float32)
ejemplos = pd.Series(["Item Name: Organic Extra Virgin Olive Oil, 16.9 Fl Oz Glass Bottle\nValue: 16.9\nUnit: Fl Oz\n",
                      "Item Name: Campbell's Condensed Chicken Noodle Soup, 10.75 Ounce Can\nValue: 10.75\nUnit: Ounce\n"])
cargado.predict(ejemplos).round(2)

array([18.2 ,  1.73])

## Conclusiones
- El modelo v2 baja el SMAPE en test de 51,3% (Ridge v1) a 46,4%. Validation y test dan casi lo mismo, así que no es suerte del split.
- Mejora en todos los tramos de precio, sobre todo en los extremos.
- Las variables que más aportan son el precio típico de la marca, el cluster de producto y el precio esperado (precio por unidad del cluster x cantidad).
- Limitaciones: sigue tirando hacia el centro (sobreestima lo muy barato y subestima lo muy caro) y hay ruido en las etiquetas.
- Siguiente paso: añadir las imágenes (notebooks 05-07).